# Kernel Leak Lab — Advanced
You are evaluating **WPInference**. Recover three confidential weights from its first convolution kernel by implementing correlation power analysis (CPA).

This version uses the same instance, packets, known first weight, three targets, acquisition budget, and defense comparisons as the beginner version. You will implement the numerical work yourself. Use the supplied `Investigation` object to access packets, track acquisitions, freeze selections, and save submissions. The analysis work areas in this notebook are left blank.

For each section, **write and run the requested code, produce the specified graphs or numerical outputs, and explain what your own results mean**. Comments below mark work areas; replace them with your implementation. Add cells and functions as needed. Choose and import your own numerical and plotting libraries. The conceptual explanations are specifications, not supplied analysis code.

In [ ]:
# Import the lab-specific interface and initialize one investigation.
from lab import Investigation
lab = Investigation()
print("Instance:", lab.manifest["instance_id"])
print("Known first weight:", lab.known)
print("Target indices:", lab.manifest["target_indices"])

## 1 · Implement fixed-width leakage calculations
A 3×3 kernel has nine weights. WPInference multiplies corresponding input and weight values and adds the nine products into an accumulator, one multiply-accumulate (MAC) at a time. Inputs and weights are signed int8 integers from −128 to 127; the accumulator is signed 32-bit. Zero point and bias are zero. The accumulator resets once per execution. Widen operands before multiplication; all nine sums fit int32. Weight indices run from 0 through 8. Weight 0 is known; recover 1, 2, and 3.

**Hamming weight (HW)** counts ones in a bit pattern. **Hamming distance (HD)** counts positions that change between two patterns: HW of their XOR. Encode states modulo 2³² before counting bits, including negative states. Python's unbounded signed representation is not the register representation.

**Write code** implementing your own 32-bit HW and HD functions. Display HW(7), HD(2,7), and HD(−1,0), including old/new/XOR patterns as 32-bit binary strings. Check your functions against additional positive, negative, unchanged, and boundary states. Explain why HW(7) and HD(2,7) differ and why −1 requires a specified width. Document the operands you tested and your expected results.

In [ ]:
# Import the libraries you choose for your implementation.
# Implement HW32 and HD32, print the required binary patterns and counts,
# and write checks covering positive, negative, unchanged, and boundary states.


## 2 · Load inputs and produce a power-trace graph
A trace is one execution's sequence of power measurements. The inputs array has one row per execution and nine columns. The traces array has the same row order and 108 columns, one for each power sample. Matching rows describe the same execution. Sample indices identify measurements, not input positions.

The guided example uses a separate kernel whose first weight is **37**. Its MAC pulses are centered at sample 6, then 18, then 30: center(j) = 6 + 12j. These are specified recording positions. The pulse center is not the MAC's start or an input index.

The supplied cell below loads the guided packet. **Write code** to inspect both shapes and display the nine inputs from one execution. **Produce a graph** of that execution's complete power trace with all nine centers marked and identified by MAC index. Label both axes, include power units (arbitrary units), and identify which execution is plotted. Explain the relation between one input row, one trace row, and the nine marked positions.

In [ ]:
# Lab-specific packet loading; the numerical inspection and plot are yours.
guided_inputs, guided_traces = lab.load_packet("guided")

In [ ]:
# Inspect and display the requested dimensions and input values.
# Plot a full guided trace with all nine labeled MAC centers.


## 3 · Implement and check the calibration correlation
For the first guided MAC, the old accumulator is zero and the new state is the first input multiplied by 37. Its predicted switching count therefore equals the HW of the fixed-width product. One prediction corresponds to one execution. Compare those predictions with sample 6 from the matching trace rows; do not correlate different sample positions within a single trace.

Pearson correlation is the sum of the products of the centered observations divided by the square root of the product of their centered sums of squares. It is dimensionless and ranges from −1 to +1. A constant vector makes the denominator zero; handle that case explicitly as undefined, rather than inventing a zero or winning score.

**Write code** to compute the predicted first-MAC switching count for every guided execution and implement Pearson correlation from its definition. Print the correlation with the matching measured samples. You may check your implementation against a library routine after writing it. Test it on positively related, negatively related, and constant vectors.

**Produce a scatter plot** with predicted switching count (bits) on the horizontal axis and measured first-MAC power (arbitrary units) on the vertical axis. Explain the sign, strength, and spread of the relationship. You are counting bits in predicted register states, not counting bits in measured power.

In [ ]:
# Compute first-MAC predictions across executions.
# Implement Pearson correlation and its zero-variance handling; run checks.
# Print the calibration result and produce the prediction-versus-power scatter plot.


## 4 · Implement candidate generation and CPA
For a target j and a candidate g, reconstruct the preceding accumulator separately for each execution r:

```text
Aold[r] = sum(x[r,k] * recovered_weight[k] for k < j)
Anew[r,g] = Aold[r] + x[r,j] * g
HD_prediction[r,g] = HW32(u32(Aold[r]) XOR u32(Anew[r,g]))
HW_prediction[r,g] = HW32(u32(Anew[r,g]))
```

The prefix contains all weights before j. For guided target 1, that prefix is [37]. There are 256 signed int8 candidates, −128 through 127 inclusive. For each candidate, correlate its prediction vector across executions with the power vector at center(j). This dataset uses positive gain, so rank by signed positive correlation. Exclude undefined correlations and explicitly handle a case where no candidate can be ranked. Candidate zero gives a constant zero HD prediction; required targets are nonzero.

**Write code** to generate candidates, reconstruct states, calculate both HW and HD prediction arrays, compute correlations with your own Pearson function, sort valid candidates, and report the top three for guided weight 1. Print each leader/runner-up gap.

**Produce correlation-versus-candidate graphs** for HW and HD with labeled axes, leakage-model identification, target index, and execution count. Explain which model matches the simulated register switching and why the first-MAC calibration could not distinguish HW from HD. A high r is not a probability of a correct weight; inspect competing candidates.

In [ ]:
# Implement the full candidate analysis for guided target 1.
# Generate both HW and HD predictions, compute and rank correlations,
# print top candidates and gaps, and plot both correlation-versus-weight curves.


## 5 · Acquire challenge observations and recover three weights
Return to the challenge: use `lab.known`, which may differ from 37. You have 320 executions: up to 256 discovery traces plus 64 reserved validation traces. All targets reuse the discovery executions.

Choose batches of 32, 64, or 128. `diverse` spans signed inputs; `narrow` uses 0…3; `repeated` uses one repeated patch. Predict the value of an acquisition before calling it. Re-running the acquisition cell spends another batch. Repeats expose measurement noise but provide no prediction variance when every input is identical. Different inputs can also produce identical candidate predictions; more copies of an unidentifiable experiment cannot resolve that ambiguity.

The supplied acquisition cell gives access to `lab.inputs`, `lab.traces`, and `lab.ledger`. **Write code** to run your own CPA on these observations. Recover targets 1, 2, and 3 sequentially using prefixes built from your selected weights. For each target, print the leader, runner-up, correlations, gap, sample center, and trace count, and **produce a candidate-correlation graph**.

Collect additional batches as justified and **produce a graph of how the correlation of selected candidates changes with discovery trace count** for at least one target (identify candidates and keep the target/prefix fixed for that comparison). Explain whether the rankings stabilized and why your measurement choices helped or failed. A wrong prefix changes all later state predictions; revisit earlier selections if needed. If the budget is exhausted with inconclusive results, retain the evidence and discuss a rescue run with your instructor.

In [ ]:
# Edit the strategy, count, and prediction before acquiring.
acquisition = lab.acquire(
    "diverse", 32,
    "Different signed inputs should provide variation in the switching predictions."
)
print(acquisition)

In [ ]:
# Analyze lab.inputs and lab.traces using your own implementations.
# Recover the three weights in order; report candidate evidence for each.
# Plot each target's candidate correlations and the requested trace-count comparison.


## 6 · Freeze selections and implement independent validation
Select your three weights using discovery data before opening the 64 fresh validation executions. The lab-specific cells freeze the tuple and open the raw observations; they perform no analysis.

**Write code** applying your own state reconstruction and correlation functions to the validation data. For each frozen target, compute its correlation and its rank among all candidates with defined correlation. Define rank as 1 plus the number of candidates with a strictly greater signed correlation. Reconstruct prefixes from your frozen tuple, not newly selected validation winners.

**Produce a graph comparing discovery and validation correlations for the three frozen weights.** Identify both datasets, label the target indices, and explain any disagreements. Report undefined correlations as Python `None`, with rank `None`. If you revise guesses after seeing validation, that batch no longer independently validates the new selection; ask the instructor about fresh data.

Create `validation_evidence` as three dictionaries with fields `index` (1, 2, 3), `chosen` (the corresponding frozen weight), `held_out_r` (a Python float or None), and `rank` (a Python int or None). The recording function checks format and stores your results; it does not calculate or verify your statistics.

In [ ]:
# Enter the three weights chosen from your discovery analysis.
guesses = []
if len(guesses) == 3:
    if lab.frozen is None:
        lab.freeze(guesses)
    print("Frozen selections:", lab.frozen)
    validation_inputs, validation_traces = lab.open_validation()
else:
    print("Select three weights before opening validation.")

In [ ]:
# Compute independent validation correlations and candidate ranks from scratch.
# Produce the discovery-versus-validation graph and build validation_evidence.


In [ ]:
# Replace this empty list with your three computed result dictionaries.
validation_evidence = []
if validation_evidence:
    if lab.validation is None:
        lab.record_validation(validation_evidence)
    print(lab.validation)

## 7 · Analyze defense comparisons
Predict what storage encryption and increased measurement noise will do to this attack. The three packets are baseline, storage encryption, and more noise. Storage encryption represents the same execution leakage after a protected model is loaded, so that packet duplicates baseline observations. It does not implement encryption. The more-noise packet increases independent noise standard deviation from 5 to 10 arbitrary units. All conditions use paired inputs; open them after freezing. They are comparison observations outside the discovery/validation budget.

**Write code** loading the three conditions with the supplied loader and running your own first-target CPA for each. Print the leading two weights, correlations, and gaps. **Produce correlation-versus-candidate graphs that compare the three conditions**, using clear labels even where curves coincide. Explain which outputs are identical by construction, what changes with noise, and why one noisy experiment cannot prove resistance. Propose a mitigation, such as carefully designed masking, and discuss an implementation cost or further evaluation needed.

In [ ]:
# Lab-specific packet loading only. Add your own numerical analysis and plots below.
if lab.frozen is not None:
    defense_packets = {
        name: lab.load_packet("defense_" + name)
        for name in ("baseline", "storage_encryption", "more_noise")
    }
else:
    print("Freeze your selections before opening defense observations.")

In [ ]:
# Analyze every condition using your own CPA and plotting code.
# Report the leader/runner-up evidence and graph all three correlation curves.


## 8 · Explain and submit
Complete each note with your own calculations, numerical outputs, and graph interpretations. The transfer question is: **A teammate recovers weight 1 incorrectly but uses the true inputs for weight 2. Why might weight 2 fail, and what experiment would you do next?**

Submit the completed advanced notebook and `submission-advanced.json`. Include all implementation code and outputs. Follow course policy for AI assistance and disclosure. This experiment models a sequential first-layer MAC lane; explain one assumption that would need revisiting on a parallel accelerator.

In [ ]:
# Replace each empty string with your evidence and explanations.
lab.notes = {
    "hd_calculation": "",
    "acquisition_observation": "",
    "candidate_evidence": "",
    "defense": "",
    "threat_model": "",
    "transfer_answer": "",
}
# The lab handles submission serialization after your validation and notes are complete.
if lab.validation is not None and all(lab.notes.values()):
    print(lab.export("submission-advanced.json"))
else:
    print("Complete your validation calculations and all six notes before exporting.")

## Implementation and output checklist
- Implement fixed-width HW/HD; display bit patterns and test results.
- Inspect data dimensions; graph a complete trace and mark all MAC centers.
- Implement Pearson correlation with undefined-case handling; test it and graph calibration predictions versus power.
- Implement candidate hypotheses, HW/HD CPA, valid-score ranking, and runner-up gaps; graph the guided results.
- Recover three challenge weights; graph each candidate ranking and a trace-count comparison.
- Compute validation scores/ranks yourself; graph discovery versus validation for the frozen weights.
- Analyze and graph all defense conditions.
- Explain your outputs and submit your implementation, figures, ledger, and evidence notes.

Use `Investigation` for packet access, acquisition tracking, and submission management.